## Verification


In [1]:
import numpy as np
import modgen2d as mg2d
import helper_functions as hf

import gc, random
import time
import psutil
import numpy as np
import pandas as pd

from memory_profiler import memory_usage

In [2]:
def run_simulation(method, x_span, z_span, run_seed):
    rng = np.random.default_rng(run_seed)

    # Generate the same type of lithological model for both methods
    main_properties_config_instance, lit_collection = (
        hf.get_properties_and_lit_domain(
            n_layers,
            x_span,
            z_span,
            gwt_depth,
            rng=rng,
            plot=False,
        )
    )

    main_properties_config_instance.lock_and_generate_sample_properties(lit_collection)

    if method == "CMD":
        spatial_sim = (
            mg2d.spatial_simulator2d.CovarianceDecompositionSimulator(
                spatial_theta_x,
                spatial_theta_z,
                rng=rng,
            )
        )

    elif method == "GSTools":
        spatial_sim = mg2d.spatial_simulator2d.GSToolsSimulator(
            spatial_theta_x,
            spatial_theta_z,
            mode_no=mode_no,
            rng=rng,
        )

    else:
        raise ValueError(f"Unknown method: {method}")

    generated_profiles = mg2d.GeneratedProfileCollection2D(
        main_properties_config_instance,
        lit_collection,
        spatial_sim,
    )

    generated_profiles.simulate_property_profile("vs")
    # generated_profiles.merged_generated_model2d.plot_profile('vs')
    return generated_profiles

In [3]:
grid_shapes = [
    (32, 16),      # 1,024 cells
    (64, 16),      # 1,024 cells
    (32, 32),      # 1,024 cells
    (128, 16),     # 2,048 cells
    (128, 32),     # 4,096 cells
    (256, 32),     # 8,192 cells
    (512, 32),   # 16,384 cells
    (512, 64),   # 32,768 cells
]

n_repeats = 5
base_seed = 42

spatial_theta_x = 100
spatial_theta_z = 0.5
mode_no = 1000

n_layers = 4
gwt_depth = 5

methods = ["CMD", "GSTools"]


In [4]:
# Warm up both simulation methods
warmup_x_span = 32
warmup_z_span = 16

print("Performing unrecorded warm-up runs...")

for method in methods:
    for warmup_index in range(2):
        warmup_result = run_simulation(
            method=method,
            x_span=warmup_x_span,
            z_span=warmup_z_span,
            run_seed=base_seed + warmup_index,
        )

        del warmup_result
        gc.collect()

print("Warm-up complete.\n")


# Runtime and memory benchmarks
raw_results = []
order_rng = np.random.default_rng(base_seed)

for grid_index, (x_span, z_span) in enumerate(grid_shapes):

    n_cells = x_span * z_span

    for repeat in range(1, n_repeats + 1):

        run_seed = (
            base_seed
            + grid_index * n_repeats
            + repeat
        )

        # Prevent either method from always running first
        method_order = list(
            order_rng.permutation(methods)
        )

        for method in method_order:

            # if method == "CMD" and n_cells > 10000:
            #     continue

            try:
                # 1. Runtime-only measurement
                gc.collect()

                start_time = time.perf_counter()

                generated_profiles = run_simulation(
                    method=method,
                    x_span=x_span,
                    z_span=z_span,
                    run_seed=run_seed,
                )

                elapsed_time = (
                    time.perf_counter()
                    - start_time
                )

                del generated_profiles
                gc.collect()

                # 2. Separate peak-memory measurement
                process = psutil.Process()

                baseline_memory_mib = (
                    process.memory_info().rss
                    / (1024**2)
                )

                peak_memory_mib, memory_result = memory_usage(
                    (
                        run_simulation,
                        (
                            method,
                            x_span,
                            z_span,
                            run_seed,
                        ),
                    ),
                    interval=0.01,
                    max_usage=True,
                    retval=True,
                    include_children=True,
                )

                additional_peak_memory_mib = max(
                    0.0,
                    peak_memory_mib
                    - baseline_memory_mib,
                )

                del memory_result
                gc.collect()

                raw_results.append(
                    {
                        "method": method,
                        "x_max": x_span,
                        "z_max": z_span,
                        "n_cells": n_cells,
                        "repeat": repeat,
                        "seed": run_seed,
                        "time_s": elapsed_time,
                        "additional_peak_memory_mib": (
                            additional_peak_memory_mib
                        ),
                        "status": "completed",
                        "error": None,
                    }
                )

            except Exception as error:

                raw_results.append(
                    {
                        "method": method,
                        "x_max": x_span,
                        "z_max": z_span,
                        "n_cells": n_cells,
                        "repeat": repeat,
                        "seed": run_seed,
                        "time_s": np.nan,
                        "additional_peak_memory_mib": np.nan,
                        "status": "failed",
                        "error": repr(error),
                    }
                )

                gc.collect()

            print(
                f"{method:7s} | "
                f"{x_span:4d} × {z_span:<4d} | "
                f"{n_cells:7,d} cells | "
                f"repeat {repeat}/{n_repeats} | "
                f"{raw_results[-1]['status']}"
            )


results_df = pd.DataFrame(raw_results)
results_df

Performing unrecorded warm-up runs...
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 4
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 4
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 4
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 4
Warm-up complete.

Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
S

,method,x_max,z_max,n_cells,repeat,seed,time_s,additional_peak_memory_mib,status,error
0,GSTools,32,16,512,1,43,0.032285,92.214844,completed,None
1,CMD,32,16,512,1,43,0.010603,91.792969,completed,None
2,CMD,32,16,512,2,44,0.012223,91.796875,completed,None
3,GSTools,32,16,512,2,44,0.032400,91.925781,completed,None
4,CMD,32,16,512,3,45,0.012823,91.796875,completed,None
...,...,...,...,...,...,...,...,...,...,...
75,CMD,512,64,32768,3,80,52.322761,10957.335938,completed,None
76,GSTools,512,64,32768,4,81,1.274395,92.773438,completed,None
77,CMD,512,64,32768,4,81,70.045189,9081.277344,completed,None
78,GSTools,512,64,32768,5,82,1.269241,91.792969,completed,None


In [5]:
completed_df = results_df.loc[
    results_df["status"] == "completed"
].copy()


summary_df = (
    completed_df
    .groupby(
        [
            "method",
            "x_max",
            "z_max",
            "n_cells",
        ],
        as_index=False,
    )
    .agg(
        median_runtime_s=("time_s", "median"),
        mean_runtime_s=("time_s", "mean"),
        std_runtime_s=("time_s", "std"),
        median_memory_mib=(
            "additional_peak_memory_mib",
            "median",
        ),
        mean_memory_mib=(
            "additional_peak_memory_mib",
            "mean",
        ),
        std_memory_mib=(
            "additional_peak_memory_mib",
            "std",
        ),
        n_successful_runs=("repeat", "count"),
    )
)


summary_df["runtime_s"] = (
    summary_df["median_runtime_s"]
    .map("{:.3f}".format)
)


summary_df["additional_peak_memory_mib"] = (
    summary_df["median_memory_mib"]
    .map("{:.1f}".format)
)


summary_df

,method,x_max,z_max,n_cells,median_runtime_s,mean_runtime_s,std_runtime_s,median_memory_mib,mean_memory_mib,std_memory_mib,n_successful_runs,runtime_s,additional_peak_memory_mib
0,CMD,32,16,512,0.011359,0.011514,0.000997,91.796875,91.795312,0.002140,5,0.011,91.8
1,CMD,32,32,1024,0.020212,0.021510,0.003038,91.792969,94.478125,3.723856,5,0.020,91.8
2,CMD,64,16,1024,0.022723,0.023524,0.003288,93.070312,96.757031,6.726382,5,0.023,93.1
3,CMD,128,16,2048,0.120116,0.124143,0.038919,155.824219,151.325000,30.991216,5,0.120,155.8
4,CMD,128,32,4096,0.485646,0.565542,0.228685,328.496094,357.928906,137.035104,5,0.486,328.5
5,CMD,256,32,8192,1.591648,1.775412,0.538683,670.218750,721.728125,224.332773,5,1.592,670.2
6,CMD,512,32,16384,12.515532,19.380661,12.279704,3488.335938,4927.769531,3097.854971,5,12.516,3488.3
7,CMD,512,64,32768,72.439633,70.724413,11.753561,10710.625000,11451.969531,2653.834834,5,72.440,10710.6
8,GSTools,32,16,512,0.032504,0.033508,0.001539,91.863281,91.931250,0.163960,5,0.033,91.9
9,GSTools,32,32,1024,0.054328,0.054422,0.000268,91.785156,91.787500,0.016937,5,0.054,91.8


In [6]:
summary_df = (
    results_df.loc[results_df["status"] == "completed"]
    .groupby(
        ["method", "x_max", "z_max", "n_cells"],
        as_index=False,
    )
    .agg(
        mean_runtime_s=("time_s", "mean"),
        std_runtime_s=("time_s", "std"),
        mean_memory_mib=("additional_peak_memory_mib", "mean"),
        std_memory_mib=("additional_peak_memory_mib", "std"),
        n_successful_runs=("repeat", "count"),
    )
)

summary_df["runtime_s"] = (
    summary_df["mean_runtime_s"].map("{:.3f}".format)
    + " ± "
    + summary_df["std_runtime_s"].map("{:.3f}".format)
)

summary_df["additional_peak_memory_mib"] = (
    summary_df["mean_memory_mib"].map("{:.1f}".format)
    + " ± "
    + summary_df["std_memory_mib"].map("{:.1f}".format)
)

summary_df

,method,x_max,z_max,n_cells,mean_runtime_s,std_runtime_s,mean_memory_mib,std_memory_mib,n_successful_runs,runtime_s,additional_peak_memory_mib
0,CMD,32,16,512,0.011514,0.000997,91.795312,0.002140,5,0.012 ± 0.001,91.8 ± 0.0
1,CMD,32,32,1024,0.021510,0.003038,94.478125,3.723856,5,0.022 ± 0.003,94.5 ± 3.7
2,CMD,64,16,1024,0.023524,0.003288,96.757031,6.726382,5,0.024 ± 0.003,96.8 ± 6.7
3,CMD,128,16,2048,0.124143,0.038919,151.325000,30.991216,5,0.124 ± 0.039,151.3 ± 31.0
4,CMD,128,32,4096,0.565542,0.228685,357.928906,137.035104,5,0.566 ± 0.229,357.9 ± 137.0
5,CMD,256,32,8192,1.775412,0.538683,721.728125,224.332773,5,1.775 ± 0.539,721.7 ± 224.3
6,CMD,512,32,16384,19.380661,12.279704,4927.769531,3097.854971,5,19.381 ± 12.280,4927.8 ± 3097.9
7,CMD,512,64,32768,70.724413,11.753561,11451.969531,2653.834834,5,70.724 ± 11.754,11452.0 ± 2653.8
8,GSTools,32,16,512,0.033508,0.001539,91.931250,0.163960,5,0.034 ± 0.002,91.9 ± 0.2
9,GSTools,32,32,1024,0.054422,0.000268,91.787500,0.016937,5,0.054 ± 0.000,91.8 ± 0.0


In [7]:
summary_df.to_csv('summary_modgen2d.csv')

In [8]:
results_df

,method,x_max,z_max,n_cells,repeat,seed,time_s,additional_peak_memory_mib,status,error
0,GSTools,32,16,512,1,43,0.032285,92.214844,completed,None
1,CMD,32,16,512,1,43,0.010603,91.792969,completed,None
2,CMD,32,16,512,2,44,0.012223,91.796875,completed,None
3,GSTools,32,16,512,2,44,0.032400,91.925781,completed,None
4,CMD,32,16,512,3,45,0.012823,91.796875,completed,None
...,...,...,...,...,...,...,...,...,...,...
75,CMD,512,64,32768,3,80,52.322761,10957.335938,completed,None
76,GSTools,512,64,32768,4,81,1.274395,92.773438,completed,None
77,CMD,512,64,32768,4,81,70.045189,9081.277344,completed,None
78,GSTools,512,64,32768,5,82,1.269241,91.792969,completed,None
